# open the tiff data

to get familiarized with it

There are some issues with the following files (Virtualizarr cannot open them)

- input_files/NSIDC-0477_AMSR_37V_CO_FT_2023_day365_v05.2.tif




In [1]:
from pathlib import Path
import json
import xarray as xr

## virtualizarr specific libraries
from virtualizarr import open_virtual_dataset
from virtual_tiff import VirtualTIFF
from obstore.store import LocalStore
from obspec_utils.registry import ObjectStoreRegistry

# pydap specific libraries
from pydap.model import DatasetType
from pydap.responses.dmr import DMRResponse
from pydap.parsers.dmr import DummyData

## Kerchunk files

## Input tif files

We work directly with the tif file.



In [64]:
directory_path = Path("./input_files/")

# Find all .json files in the top-level directory
tif_files = list(directory_path.glob("*.tif"))
print("found: ", len(tif_files), " tif files")
filename = tif_files[6]
filename

found:  7  tif files


PosixPath('input_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif')

## Virtual datastore

The end result is an Xarray Dataset object. Will use an intermediate product, to translate the information into the pydap data model


In [65]:
filepath = f"{filename.resolve().parent}/{filename.name}"

registry = ObjectStoreRegistry({"file://": LocalStore()})
parser = VirtualTIFF(ifd_layout="nested")

ms = parser(f"file://{filepath}", registry=registry)
ds = ms.to_virtual_datatree()
ds

/Users/jimenezm/miniforge3/envs/dmrpp_tests/lib/python3.12/site-packages/virtual_tiff/parser.py:211: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  return codec()
/Users/jimenezm/miniforge3/envs/dmrpp_tests/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:77: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  return cls(**data.get("configuration", {}))


<xarray.DataTree>
Group: /
├── Group: /0
│       Dimensions:  (y: 2326, x: 2171)
│       Dimensions without coordinates: y, x
│       Data variables:
│           0        (y, x) float32 20MB ManifestArray<shape=(2326, 2171), dtype=floa...
│       Attributes: (12/14)
│           citation:                    WGS 84 / UTM zone 11N
│           geog_angular_units:          9102
│           geog_citation:               WGS 84
│           model_type:                  1
│           proj_linear_units:           9001
│           projected_type:              32611
│           ...                          ...
│           model_tiepoint:              [0.0, 0.0, 0.0, 360574.992247703, 3799093.19...
│           photometric_interpretation:  1
│           Band_1:                      Band 1
│           DESCRIPTION:                 Band 1
│           gdal_no_data:                -9999
│           _FillValue:                  AAAAAICHw8A=
├── Group: /1
│       Dimensions:  (y: 1163, x: 1085)
│       Dimensions without coordinates: y, x
│       Data variables:
│           1        (y, x) float32 5MB ManifestArray<shape=(1163, 1085), dtype=float...
│       Attributes:
│           photometric_interpretation:  1
│           gdal_no_data:                -9999
│           _FillValue:                  AAAAAICHw8A=
├── Group: /2
│       Dimensions:  (y: 581, x: 542)
│       Dimensions without coordinates: y, x
│       Data variables:
│           2        (y, x) float32 1MB ManifestArray<shape=(581, 542), dtype=float32...
│       Attributes:
│           photometric_interpretation:  1
│           gdal_no_data:                -9999
│           _FillValue:                  AAAAAICHw8A=
└── Group: /3
        Dimensions:  (y: 290, x: 271)
        Dimensions without coordinates: y, x
        Data variables:
            3        (y, x) float32 314kB ManifestArray<shape=(290, 271), dtype=float...
        Attributes:
            photometric_interpretation:  1
            gdal_no_data:                -9999
            _FillValue:                  AAAAAICHw8A=

## Parsing metadata

The following steps will enable the creation of a pydap dataset, and with it, the generation of a dmrpp



In [66]:
groups = list(ms._group.groups.keys())
groups

['0', '1', '2', '3']

In [67]:
ms._group.arrays # root arrays?

{}

In [68]:
ms._group.metadata.attributes

{}

## Hierarchical data

Extract any nested data and place it within Groups


In [69]:
GROUPS = {}
global_attrs = {}
for group in groups:
    arrays={}
    for k,v in ms._group[group].arrays.items():
        if k=='0':
            global_attrs=ms._group[group].metadata.attributes
        chunk_manifest = {"chunk_shape": v.metadata.chunk_grid.chunk_shape, "fill_value": v.metadata.fill_value, "codecs": v.metadata.codecs,"hrefs": v.manifest.dict()}
        arrays.update({k:{'shape': v.shape, 'dtype': v.dtype, "dims": v.metadata.dimension_names, "chunk_manifest": chunk_manifest}})
    GROUPS.update({group:{"attributes": {**global_attrs, **ms._group[group].metadata.attributes}, "arrays": arrays}})

The nested dictionary holds information about the array data that will go into the dmrpp

## Populate the pydap dataset

Here, the virtualizarr metadata is injected into the pydap data model


In [70]:
def array_metadata(arraymeta: dict, parent: str | None = None):
    """Reads array medatata extracteed from virtualizarr, and
    and re structures it following pydap-specific syntax
    """
    if not parent:
        parent = "/"
    _dims_shapes = dict((dim, size) for dim, size in zip(arraymeta['dims'], arraymeta['shape']))
    _dims = ["/".join([parent,dim]) for dim in list(_dims_shapes)]
    _data = DummyData(dtype= arraymeta['dtype'], shape= arraymeta['shape'], path = parent)
    return _dims, _dims_shapes, _data

In [71]:
pyds = DatasetType(name=filename.name, attributes=ms._group.metadata.attributes)
for array in ms._group.arrays:
    dims = ms._group[array].metadata.dimensions_names
    data = DummyData(dtype=ms._group[array].dtype, shape=ms._group[array].shape, path='/')
    pyds.createVariable(name=array, dims=dims, data=data)

### Now all hierarchical data
for gr in GROUPS:
    _DIMS = {}
    group_name = "/" + gr
    pyds.createGroup(name=group_name, attributes=GROUPS[gr]['attributes'])
    for array in GROUPS[gr]["arrays"]:
        var_name = "/".join([group_name,array])
        dims, dim_shapes, data = array_metadata(arraymeta = GROUPS[gr]["arrays"][array], parent=group_name)
        pyds.createVariable(name = var_name, dims = dims, data=data)
        _DIMS.update(dim_shapes)
    pyds[group_name].attributes['dimensions'] = _DIMS

In [72]:
dmr_name = f"./output_files/{filename.name}.dmr"
dmr_name

'./output_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif.dmr'

## Write DMR to disk

In [73]:
with open(dmr_name, "wb") as file:
    file.write(b"".join(DMRResponse(pyds)).decode("ascii").encode("utf-8"))

In [74]:
GROUPS['0']['attributes']

{'citation': 'WGS 84 / UTM zone 11N',
 'geog_angular_units': 9102,
 'geog_citation': 'WGS 84',
 'model_type': 1,
 'proj_linear_units': 9001,
 'projected_type': 32611,
 'raster_type': 1,
 'model_pixel_scale': [2.9, 2.9, 0.0],
 'model_tiepoint': [0.0, 0.0, 0.0, 360574.992247703, 3799093.19969418, 0.0],
 'photometric_interpretation': <PhotometricInterpretation.BlackIsZero: 1>,
 'Band_1': 'Band 1',
 'DESCRIPTION': 'Band 1',
 'gdal_no_data': '-9999',
 '_FillValue': 'AAAAAICHw8A=',
 'path': '/',
 'dimensions': {'y': 2326, 'x': 2171}}

In [75]:
GROUPS['0']['arrays']['0']['chunk_manifest'].keys()

dict_keys(['chunk_shape', 'fill_value', 'codecs', 'hrefs'])

In [76]:
GROUPS['0']['arrays']['0']['chunk_manifest']['chunk_shape']

(512, 512)

In [77]:
GROUPS['0']['arrays']['0']['chunk_manifest']['fill_value']

np.float32(-9999.0)

In [78]:
GROUPS['0']['arrays']['0']['chunk_manifest']['codecs']

(BytesCodec(endian='little'),
 LZWCodec(codec_name='imagecodecs_lzw', _codec=Lzw(), codec_config={'id': 'imagecodecs_lzw'}))

In [79]:
GROUPS['0']['arrays']['0']['chunk_manifest']['hrefs']

{'0.0': {'path': 'file:///Users/jimenezm/kerchunk-dmrpp/input_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif',
  'offset': 3567918,
  'length': 81219},
 '0.1': {'path': 'file:///Users/jimenezm/kerchunk-dmrpp/input_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif',
  'offset': 3649145,
  'length': 845106},
 '0.2': {'path': 'file:///Users/jimenezm/kerchunk-dmrpp/input_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif',
  'offset': 4494259,
  'length': 404206},
 '0.3': {'path': 'file:///Users/jimenezm/kerchunk-dmrpp/input_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif',
  'offset': 4898473,
  'length': 3994},
 '0.4': {'path': 'file:///Users/jimenezm/kerchunk-dmrpp/input_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif',
  'offset': 4902475,
  'length': 4470},
 '1.0': {'path': 'file:///Users/jimenezm/kerchunk-dmrpp/input_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif',
  'offset': 4906953,
  'length': 950178},
 '1.1': {'path': 'fil

## Translating `chunk_manifest` into `dmrpp:chunks` XML

Translates the zarr-style chunk manifest (`chunk_shape`, `fill_value`, `codecs`, `hrefs`) extracted per array into the `<dmrpp:chunks>` / `<dmrpp:chunk>` XML fragment used inside a dmrpp file.

Zarr chunk keys such as `"1.2"` are converted into DAP's `chunkPositionInArray`, e.g. for `chunk_shape=(256,256)`, key `"1.2"` -> `[256, 512]` (index * chunk_shape per dimension).

In [80]:
def _chunk_key_to_position(key: str, chunk_shape: tuple) -> list:
    """Translate a zarr chunk key (e.g. '1.2') into the dmrpp
    `chunkPositionInArray` (e.g. [256, 512] for chunk_shape=(256,256)).
    """
    indices = [int(i) for i in key.split(".")]
    return [idx * size for idx, size in zip(indices, chunk_shape)]


# maps zarr/numcodecs/imagecodecs compressor names onto dmrpp compressionType values
_COMPRESSOR_TO_DMRPP = {
    "numcodecs.zlib": "deflate",
    "numcodecs.gzip": "deflate",
    "imagecodecs_lzw": "lzw",
}


def _codec_attributes(codecs) -> dict:
    """Inspect a zarr/virtual-tiff codecs pipeline and pull out the bits that
    map onto `dmrpp:chunks` attributes: byteOrder, compressionType and
    deflateLevel. Filter/predictor codecs (delta, transpose, float-predictor,
    etc.) don't have a dmrpp equivalent and are ignored.
    """
    attrs = {"byteOrder": "LE"}
    for codec in codecs:
        endian = getattr(codec, "endian", None)
        if endian is not None:
            attrs["byteOrder"] = "BE" if "big" in str(endian).lower() else "LE"
        codec_name = getattr(codec, "codec_name", None)
        if codec_name in _COMPRESSOR_TO_DMRPP:
            attrs["compressionType"] = _COMPRESSOR_TO_DMRPP[codec_name]
            level = getattr(codec, "codec_config", {}).get("level")
            if level is not None:
                attrs["deflateLevel"] = level
    return attrs


## Writing the `.dmrpp` file

Insert each array's `<dmrpp:chunks>` element into a copy of the generated `.dmr` file, matching by group name and array name, and write the result next to the source `.dmr` with a `.dmrpp` extension.

In [81]:
import xml.etree.ElementTree as ET

DAP4_NS = "http://xml.opendap.org/ns/DAP/4.0#"
DMRPP_NS = "http://xml.opendap.org/dap/dmrpp/1.0.0#"
ET.register_namespace("", DAP4_NS)
ET.register_namespace("dmrpp", DMRPP_NS)


def _build_dmrpp_chunks_element(chunk_manifest: dict, dmrpp_href: str) -> ET.Element:
    """Build the `<dmrpp:chunks>` element (with its `<dmrpp:chunk>` children)
    for one array's chunk_manifest, ready to be appended to that array's
    variable element in a parsed DMR tree.
    """
    chunk_shape = chunk_manifest["chunk_shape"]
    fill_value = chunk_manifest["fill_value"]
    hrefs = chunk_manifest["hrefs"]

    codec_attrs = _codec_attributes(chunk_manifest["codecs"])

    attrib = {}
    if "compressionType" in codec_attrs:
        attrib["compressionType"] = codec_attrs["compressionType"]
    if "deflateLevel" in codec_attrs:
        attrib["deflateLevel"] = str(codec_attrs["deflateLevel"])
    if fill_value is not None:
        attrib["fillValue"] = str(fill_value)
    attrib["byteOrder"] = codec_attrs["byteOrder"]

    chunks_el = ET.Element(f"{{{DMRPP_NS}}}chunks", attrib)
    dim_sizes_el = ET.SubElement(chunks_el, f"{{{DMRPP_NS}}}chunkDimensionSizes")
    dim_sizes_el.text = " ".join(str(s) for s in chunk_shape)

    for key in sorted(hrefs, key=lambda k: [int(i) for i in k.split(".")]):
        chunk = hrefs[key]
        position = "[" + ",".join(str(p) for p in _chunk_key_to_position(key, chunk_shape)) + "]"
        if chunk["path"].split("/")[-1] != dmrpp_href.split("/")[-1]:
            print(chunk["path"])
            print(dmrpp_href)
            ET.SubElement(chunks_el, f"{{{DMRPP_NS}}}chunk", {
                "offset": str(chunk["offset"]),
                "nBytes": str(chunk["length"]),
                "chunkPositionInArray": position,
                "href": chunk["path"],
            })
        else:
            ET.SubElement(chunks_el, f"{{{DMRPP_NS}}}chunk", {
                "offset": str(chunk["offset"]),
                "nBytes": str(chunk["length"]),
                "chunkPositionInArray": position,
            })          
    return chunks_el


def _find_variable_element(root: ET.Element, group_name: str, array_name: str) -> ET.Element:
    """Locate the DAP4 variable element for `array_name` inside the
    top-level Group named `group_name` (e.g. group_name="0", array_name="0").
    """
    group_el = root.find(f"{{{DAP4_NS}}}Group[@name='{group_name}']")
    if group_el is None:
        raise ValueError(f"Group '{group_name}' not found in DMR")
    for child in group_el:
        if child.tag == f"{{{DAP4_NS}}}Dimension":
            continue
        if child.get("name") == array_name:
            return child
    raise ValueError(f"Variable '{array_name}' not found in group '{group_name}'")


def add_chunk_manifests_to_dmr(dmr_path, groups: dict, output_path=None) -> Path:
    """Read a `.dmr` file, insert each array's `dmrpp:chunks` element (built
    from `groups[group]["arrays"][array]["chunk_manifest"]`) into its
    matching variable, and write the result to a `.dmrpp` file next to the
    source `.dmr`.

    Parameters
    ----------
    dmr_path : str | Path
        Path to the source `.dmr` file (as written by `DMRResponse`).
    groups : dict
        The `GROUPS` dictionary, i.e.
        `{group_name: {"arrays": {array_name: {"chunk_manifest": {...}}}}}`.
    output_path : str | Path, optional
        Defaults to `dmr_path` with its suffix replaced by `.dmrpp`.

    Returns
    -------
    Path to the written `.dmrpp` file.
    """
    dmr_path = Path(dmr_path)
    tree = ET.parse(dmr_path)
    root = tree.getroot()
    # force to point to local file
    # this will need to change to admissible location
    root.set("dmrpp:href", f"file:///usr/share/hyrax/{root.get("name")}")
    

    for group_name, group_data in groups.items():
        for array_name, array_data in group_data["arrays"].items():
            var_el = _find_variable_element(root, group_name, array_name)
            var_el.append(_build_dmrpp_chunks_element(array_data["chunk_manifest"], root.attrib['dmrpp:href']))

    ET.indent(tree, space="    ")
    output_path = Path(output_path) if output_path else dmr_path.with_suffix(".dmrpp")
    tree.write(output_path, encoding="ISO-8859-1", xml_declaration=True)
    return output_path

In [82]:
dmrpp_name = add_chunk_manifests_to_dmr(dmr_name, GROUPS)
dmrpp_name

PosixPath('output_files/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif.dmrpp')

In [83]:
print(dmrpp_name.read_text())

<?xml version='1.0' encoding='ISO-8859-1'?>
<Dataset xmlns="http://xml.opendap.org/ns/DAP/4.0#" xmlns:dmrpp="http://xml.opendap.org/dap/dmrpp/1.0.0#" xml:base="http://localhost:8001" dapVersion="4.0" dmrVersion="1.0" name="AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif" dmrpp:href="file:///usr/share/hyrax/AV320250917t200534_002_L2B_GHG_f0e699cb_CH4_ORT.tif">
    <Group name="0">
        <Dimension name="y" size="2326" />
        <Dimension name="x" size="2171" />
        <Float32 name="0">
            <Dim name="/0/y" />
            <Dim name="/0/x" />
            <dmrpp:chunks compressionType="lzw" fillValue="-9999.0" byteOrder="LE">
                <dmrpp:chunkDimensionSizes>512 512</dmrpp:chunkDimensionSizes>
                <dmrpp:chunk offset="3567918" nBytes="81219" chunkPositionInArray="[0,0]" />
                <dmrpp:chunk offset="3649145" nBytes="845106" chunkPositionInArray="[0,512]" />
                <dmrpp:chunk offset="4494259" nBytes="404206" chunkPositionInArray="